# Precision provenance — dtype flow of the surface validation data

Standalone reference notebook (no pipeline run; reads existing
stores lazily).  Documents where float32/float64 enter the pipeline,
and why the store-vs-live consistency checks show residuals of
~3e-8 for some fields and exactly 0.0 for others.

**Summary of the dtype flow**

1. **Raw model data (OSN kerchunk + LLC_SURF stores): float32.**
   MITgcm LLC4320 output is distributed in single precision
   (big-endian `>f4`); the kerchunk references re-describe those
   bytes.  Verified empirically below.
2. **Calculations: dtype-preserving, with one exception.**
   numpy/xarray arithmetic keeps float32, so e.g. the Theta/Salt
   gradient chains are float32 end-to-end.  The exception is the
   JMD95 equation of state: `calculate_fields.potential_density`
   calls it via `apply_ufunc(..., output_dtypes=[float])` → float64,
   so everything downstream of density (σ₀, b, ∇b, gradb2, gradrho2,
   turner_angle) is COMPUTED in float64.
3. **The product store: always float32.**
   `zarr_dataset_global.py` casts at write time
   (`data.astype(np.float32)`), matching the input precision —
   nothing real is lost, and storage stays half the size of float64.
4. **Live-computed intermediates in the validation notebooks:**
   keep their computation dtype (float64 for the density family) —
   they are never written anywhere.

**Consequences observed in the validation notebooks**

- Consistency checks: fields whose chains are float32 in BOTH paths
  (gradtheta2, gradsalt2) agree to exactly 0.0; the density family
  is float64 live vs float32 store → residuals at 1 ULP of float32
  (eps32 = 2⁻²⁴ ≈ 5.96e-8), a few ULPs for multi-op chains.  This is
  rounding, not a stencil/grid discrepancy — a one-pixel shift would
  give O(1) relative error at this resolution (demo below).
- The white specks in the grad*2 zoom maps sit at ~1e-14 °C² m⁻²,
  ≈50 float32 quanta above the precision floor (~4e-18): resolved,
  physical near-zero gradients at critical points of the field.
- `REL_TOL` in the consistency cells cannot meaningfully go below
  ~1e-6: eps32 is the store's noise floor.

## 1. Raw data dtype (empirical)

In [ ]:
# Open the raw OSN snapshot lazily (metadata only — fast) and print
# the on-disk dtypes.  Generated by LH and Claude.
PIPELINE = "SURF"
DATE     = "2012-11-09 12:00:00"

from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid

ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["Theta", "Salt", "Eta", "U", "V", "W"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}")
for v in ["Theta", "Salt", "Eta", "U", "V", "W"]:
    print(f"  raw {v:6s}: {ds_merge[v].dtype}")

## 2. Computation dtypes (where float64 enters)

In [ ]:
# The gradient of a float32 tracer stays float32; anything through
# JMD95 becomes float64.  Generated by LH and Claude.
import dbof.preprocessing.calculate_fields as calculate_fields
import dbof.utils.native_gradient as ng

gTh = ng.calculate_native_gradient_tracer(
    ds_merge.Theta, ds_merge, grid=xgrid)
rho = calculate_fields.potential_density(ds_merge)
gR = ng.calculate_native_gradient_tracer(rho, ds_merge, grid=xgrid)

print(f"dTheta_dx (float32 chain)   : {gTh[0].dtype}")
print(f"rho_theta (JMD95, promoted) : {rho.dtype}")
print(f"drho_dx  (inherits float64) : {gR[0].dtype}")

## 3. Store dtype (writer cast)

In [ ]:
# The zarr writer casts every channel to float32
# (zarr_dataset_global.py: data.astype(np.float32)).
# Generated by LH and Claude.
import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

defn = get_subset_definition(PIPELINE, "frontal_structure")
fs, _ = filesystems.create_s3_filesystems(
    "https://s3-west.nrp-nautilus.io")
reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket="dbof", folder="surface_fields",
    run_id="field_validation_v1", dataset_name=defn["dataset_name"],
    date_prefix="20121109_120000", fs=fs,
)
arr = reader.get_channel_snapshot("gradrho2")
print(f"store gradrho2 dtype: {arr.dtype}   (computed float64, "
      "stored float32)")
del arr

## 4. Demo — why ~3e-8 is rounding, not a grid shift

Three regimes: identical float32 arithmetic (exactly 0), float64
computation rounded to float32 at one point (≈eps32), and a
one-pixel shift (order 1).  The consistency-check residuals sit in
the second regime, seven orders of magnitude below the third.

In [ ]:
# Self-contained numpy demonstration.  Generated by LH and Claude.
import numpy as np

rng = np.random.default_rng(0)
f64 = np.abs(rng.standard_normal((1000, 1000)))**2 * 1e-8

store = f64.astype(np.float32)          # float32 store vs f64 live
rel_round = np.max(np.abs(store.astype(np.float64) - f64)) / f64.max()

a32 = f64.astype(np.float32)            # identical f32 both paths
rel_same = np.max(np.abs(a32 - a32))

shifted = np.roll(f64, 1, axis=1)       # one-pixel grid shift
rel_shift = np.max(np.abs(shifted - f64)) / f64.max()

print(f"identical float32 arithmetic : {rel_same:.2e}")
print(f"float32 rounding (1 ULP)     : {rel_round:.2e}  "
      f"(eps32 = {2**-24:.2e})")
print(f"one-pixel shift              : {rel_shift:.2e}")

**Cross-references** — consistency-check results interpreted here
live in the closing cell of each subset notebook; the white-speck
analysis is in `frontal_structure.ipynb` §5.3 (gradtheta2 zoom).
Store writer cast: `src/dbof/global_dataset_creation/
zarr_dataset_global.py`.  JMD95 promotion:
`calculate_fields.potential_density` (`output_dtypes=[float]`).